# Image Classification using CNN

Image classification is the task of assigning a label to an image from a fixed set of categories — basically teaching a computer to look at a picture and say "that's a cat" or "that's a car." It's one of the most fundamental problems in computer vision.

CNNs (Convolutional Neural Networks) are the go-to architecture for this because they're designed to work directly with pixel data. Instead of flattening an image into a huge vector and losing all spatial information, CNNs learn to detect patterns — edges, textures, shapes — by sliding small filters across the image. This makes them both efficient and powerful for visual tasks.

## Key Components of a CNN

Before jumping into code, it helps to understand what's actually happening inside the network.

- **Convolutional Layers** — These are the core building blocks. They apply learned filters to the input, producing feature maps that highlight things like edges or color gradients. Each filter learns to detect a specific pattern.

- **ReLU Activation** — After each convolution, we apply ReLU (Rectified Linear Unit), which just replaces any negative values with zero. This introduces non-linearity, which is what allows the network to learn complex relationships rather than just linear ones.

- **Pooling Layers** — These downsample the feature maps, usually by taking the max value in a small region (Max Pooling). This reduces the spatial dimensions, cutting computation and also making the features more robust to small shifts in the image.

- **Fully Connected Layers** — After the convolutional layers have extracted features, we flatten everything and pass it through dense layers. This is where the high-level reasoning happens and class scores start forming.

- **Softmax Output** — The final layer applies Softmax to convert raw scores into probabilities across all classes. The class with the highest probability becomes the model's prediction.

## How CNNs Work for Image Classification

There are three main stages that happen when a CNN processes an image:

**1. Preprocessing the Image**  
Raw images come in all shapes and sizes, and pixel values can range from 0 to 255. Before feeding anything into the network, we resize images to a consistent shape and normalize pixel values to the [0, 1] range. This makes training more stable and helps the optimizer converge faster.

**2. Feature Extraction**  
This is where the convolutional and pooling layers do their thing. Early layers tend to pick up on low-level features like edges and corners. Deeper layers combine those into more complex structures — textures, parts of objects, and eventually whole object shapes. The network learns all of this automatically from the training data, which is one of the biggest advantages over older hand-crafted feature methods.

**3. Classification**  
Once the feature extraction is done, the flattened feature vector is passed through fully connected layers. The final Softmax layer outputs a probability distribution over all classes, and we pick the one with the highest score as the predicted label.

## Step 1 — Import Libraries

In [ ]:
import tensorflow as tf                                      # main deep learning framework
from tensorflow.keras import layers, models, datasets        # layers for building the model, datasets for CIFAR-10
import matplotlib.pyplot as plt                              # for plotting training curves

## Step 2 — Load and Prepare the CIFAR-10 Dataset

CIFAR-10 is a well-known benchmark dataset containing 60,000 color images at 32×32 pixels, split across 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, and truck. It's a solid dataset for getting started with image classification — small enough to train on a laptop but complex enough to be a real challenge.

Before training, I'm doing two things to the data:
- **Scaling**: dividing pixel values by 255 to bring them into the [0, 1] range. Neural networks train much better with small, normalized inputs.
- **One-hot encoding**: converting the integer class labels (e.g., `3`) into binary vectors (e.g., `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`). This is required because the output layer uses Softmax, which outputs a probability per class.

In [ ]:
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()  # load CIFAR-10 train and test splits
x_train, x_test = x_train / 255.0, x_test / 255.0                    # normalize pixel values from [0,255] to [0,1]
num_classes = 10                                                       # CIFAR-10 has exactly 10 categories
y_train = tf.keras.utils.to_categorical(y_train, num_classes)         # one-hot encode training labels
y_test  = tf.keras.utils.to_categorical(y_test,  num_classes)         # one-hot encode test labels

## Step 3 — Build the CNN Model

The architecture here is fairly standard for a small image classification task. I'm stacking three convolutional blocks, each using a 3×3 filter with `same` padding so the spatial dimensions don't shrink from the convolution itself. The first two blocks are followed by max pooling to downsample, while the third feeds directly into the flatten layer. After flattening, a dense layer with ReLU does the high-level reasoning, and the final Softmax layer outputs class probabilities.

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),  # first conv layer: 32 filters, 3x3 kernel, ReLU, input is 32x32 RGB
    layers.MaxPooling2D(2,2),                                                            # halve spatial dims from 32x32 to 16x16
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),                         # second conv layer: 64 filters to learn more complex features
    layers.MaxPooling2D(2,2),                                                            # halve spatial dims from 16x16 to 8x8
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),                         # third conv layer: 64 filters, no pooling after this one
    layers.Flatten(),                                                                    # flatten 8x8x64 feature maps into a 1D vector
    layers.Dense(64, activation='relu'),                                                 # fully connected layer for high-level feature combination
    layers.Dense(num_classes, activation='softmax')                                      # output layer: one probability per class
])
model.summary()                                                                          # print a summary of the model architecture

## Step 4 — Compile and Train the Model

I'm using Adam as the optimizer — it's adaptive, meaning it adjusts the learning rate for each parameter individually, which usually leads to faster and more stable convergence than vanilla SGD. Categorical crossentropy is the right loss function here since we're doing multi-class classification with one-hot encoded labels. It measures how far the predicted probability distribution is from the true one-hot distribution.

In [ ]:
model.compile(optimizer='adam',                    # Adam handles adaptive learning rates automatically
              loss='categorical_crossentropy',     # correct loss for multi-class one-hot targets
              metrics=['accuracy'])                # track accuracy alongside loss during training

history = model.fit(x_train, y_train,             # train on the full training set
                    epochs=15,                     # run for 15 full passes over the data
                    batch_size=64,                 # process 64 images per gradient update
                    validation_split=0.2,          # hold out 20% of training data for validation each epoch
                    verbose=2)                     # print one line per epoch instead of a progress bar

## Step 5 — Evaluate the Model

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)  # evaluate on the held-out test set, suppress per-batch output
print(f"Test accuracy = {test_acc:.3f}")                          # print final test accuracy rounded to 3 decimal places

## Step 6 — Plot Training vs Validation Accuracy

In [ ]:
plt.plot(history.history['accuracy'], label='train')        # plot training accuracy across epochs
plt.plot(history.history['val_accuracy'], label='val')      # plot validation accuracy across epochs
plt.legend()                                                # show legend to distinguish the two lines
plt.title('Accuracy')                                       # title for the plot
plt.xlabel('Epoch')                                         # x-axis label
plt.ylabel('Accuracy')                                      # y-axis label
plt.grid(True)                                              # add a grid for easier reading
plt.show()                                                  # render and display the plot

## Benefits & Challenges of CNNs for Image Classification

**Benefits**
- **Automatic feature learning** — No need to hand-engineer features; the network figures out what matters directly from the data.
- **Translation invariance** — Pooling layers make the model less sensitive to exactly where in the image an object appears.
- **Computationally efficient via pooling** — Downsampling reduces the number of parameters and computation needed in later layers.
- **Scales well with large datasets** — More data generally means better generalization, and CNNs are well-suited to take advantage of it.

**Challenges**
- **Overfitting with limited data** — CNNs have a lot of parameters; without enough training examples they can memorize rather than generalize.
- **High computational demands** — Training deep CNNs is resource-intensive and often requires a GPU to be practical.
- **Requires quality labeled data** — The model is only as good as its labels; noisy or mislabeled data directly hurts performance.
- **Long training times** — Even with a GPU, training for many epochs on large datasets can take hours or days.